In [20]:
!pip install pydantic

In [21]:
from google.colab import drive
drive.mount('/content/drive')
nurses_cleaned_data_path="https://drive.google.com/uc?id=1MW9XN4xwwFR6I6fV3oudl6IRTu9jMLwP"
nurses_merged_deduplicated_data_path="https://drive.google.com/uc?id=1LRgF_QS2vN4d5UeTp1t9I-MwqET_OAuH"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
from pydantic import BaseModel, EmailStr, field_validator
from typing import Optional

# This is the "Brain" or Template for a polished record
class PolishedNurse(BaseModel):
    first_name: str
    last_name: str
    job_title: Optional[str] = "N/A"
    email: Optional[str] = None
    npi_number: Optional[str] = None

    # REFINEMENT 1: Automatically capitalize names
    @field_validator('first_name', 'last_name')
    @classmethod
    def capitalize_names(cls, v: str) -> str:
        return v.strip().title()

    # REFINEMENT 2: Automatically lowercase emails
    @field_validator('email')
    @classmethod
    def email_to_lowercase(cls, v: str) -> str:
        if v:
            return v.strip().lower()
        return v

# --- Let's test it with "Messy" Data ---
raw_data = {
    "first_name": "  satya",
    "last_name": "MAMIDI",
    "job_title": "icu nurse",
    "email": "SATYA@GMAIL.COM"
}

# Run the raw data through the polisher
polished_record = PolishedNurse(**raw_data)

print("--- BEFORE (Messy) ---")
print(raw_data)
print("\n--- AFTER (Polished) ---")
print(polished_record.dict())

--- BEFORE (Messy) ---
{'first_name': '  satya', 'last_name': 'MAMIDI', 'job_title': 'icu nurse', 'email': 'SATYA@GMAIL.COM'}

--- AFTER (Polished) ---
{'first_name': 'Satya', 'last_name': 'Mamidi', 'job_title': 'icu nurse', 'email': 'satya@gmail.com', 'npi_number': None}


/tmp/ipykernel_699/1602775098.py:40: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(polished_record.dict())


In [23]:
import pandas as pd
import io

# Let's create a small version of your dataset to practice on
csv_content = """first_name,last_name,job_title,email
elizabeth,BROWN,ICU Nurse,ELIZABETH.B@EXAMPLE.COM
john,smith,Nurse Anesthetist,J.Smith@clinic.org
"""

df = pd.read_csv(io.StringIO(csv_content))

# This function uses your "Brain" to polish each row
def polish_row(row):
    try:
        p = PolishedNurse(
            first_name=row['first_name'],
            last_name=row['last_name'],
            job_title=row['job_title'],
            email=row['email']
        )
        return p.dict()
    except Exception as e:
        return {"error": str(e)}

# Apply the polishing
polished_df = pd.DataFrame([polish_row(row) for _, row in df.iterrows()])

print("Your Polished Dataset:")
polished_df.head()

Your Polished Dataset:


/tmp/ipykernel_699/1549117701.py:21: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return p.dict()


,first_name,last_name,job_title,email,npi_number
0,Elizabeth,Brown,ICU Nurse,elizabeth.b@example.com,None
1,John,Smith,Nurse Anesthetist,j.smith@clinic.org,None


In [24]:
import pandas as pd
from pydantic import BaseModel, field_validator, ConfigDict
from typing import Optional

# 1. THE BRAIN: Updated to match your actual CSV columns
class HealthCrewPolisher(BaseModel):
    # This allows the model to accept extra columns without crashing
    model_config = ConfigDict(extra='ignore')

    first_name: str
    last_name: str
    email: Optional[str] = None
    job_title: Optional[str] = "N/A"
    company: Optional[str] = "N/A"
    phone: Optional[str] = None

    # REFINEMENT: Fixes 'elizabeth' -> 'Elizabeth'
    @field_validator('first_name', 'last_name')
    @classmethod
    def format_names(cls, v: str) -> str:
        return v.strip().title() if v else v

    # REFINEMENT: Fixes 'SATYA@GMAIL.COM' -> 'satya@gmail.com'
    @field_validator('email')
    @classmethod
    def format_email(cls, v: str) -> str:
        return v.strip().lower() if v else v

# 2. LOAD YOUR ACTUAL DATA
# Make sure you uploaded 'nurses_cleaned.csv' to the Colab sidebar
try:
    df_raw = pd.read_csv(nurses_cleaned_data_path)
    print("Successfully loaded nurses_cleaned.csv")
except FileNotFoundError:
    print("Error: Please upload nurses_cleaned.csv to the sidebar folder first!")

# 3. THE REFINERY FUNCTION
def polish_nurse_data(row):
    try:
        # We map your CSV column names to our Polisher logic
        nurse = HealthCrewPolisher(
            first_name=row['first_name'],
            last_name=row['last_name'],
            email=row.get('email_address'), # Your CSV uses 'email_address'
            job_title=row.get('job_title'),
            company=row.get('company'),
            phone=row.get('phone_number')
        )
        return nurse.model_dump()
    except Exception as e:
        return None

# 4. RUN THE TRANSFORMATION
polished_data = [polish_nurse_data(row) for _, row in df_raw.head(20).iterrows()]
df_final = pd.DataFrame([p for p in polished_data if p])

print("\n--- POLISHED DATA PREVIEW ---")
df_final.head()

Successfully loaded nurses_cleaned.csv

--- POLISHED DATA PREVIEW ---


""


In [25]:
import pandas as pd

# Load the raw messy input and the current "best" output
df_raw = pd.read_csv(nurses_cleaned_data_path)
df_final = pd.read_csv(nurses_merged_deduplicated_data_path)

print(f"Raw Rows: {len(df_raw)} | Final Rows: {len(df_final)}")
# Look at a specific person to see how their data was 'polished'
display(df_raw[df_raw['last_name'] == 'Brown'].head(2))
display(df_final[df_final['last_name'] == 'Brown'].head(2))

Raw Rows: 5000 | Final Rows: 5000


,first_name,last_name,email_address,job_title,company,location,city,state,country,phone_number,linkedin_url,facebook_url,gender
0,Elizabeth,Brown,NaN,ICU Nurse,Valley Medical Center,"Birmingham, AL",Birmingham,AL,USA,450-428-3286,https://www.linkedin.com/in/elizabeth-brown/,https://www.facebook.com/100000000001824,NaN
1,Barbara,Brown,NaN,Certified Registered Nurse Anesthetist,HCA Healthcare,"Birmingham, AL",Birmingham,AL,USA,232-230-2535,https://www.linkedin.com/in/barbara-brown/,https://www.facebook.com/100000000012286,M


,first_name,last_name,email_address,job_title,company,location,city,state,country,phone_number,linkedin_url,facebook_url,gender
0,Elizabeth,Brown,elizabeth.brown0@nurse-sample.example.com,ICU Nurse,Valley Medical Center,"Birmingham, AL",Birmingham,AL,USA,450-428-3286,https://www.linkedin.com/in/elizabeth-brown/,https://www.facebook.com/100000000001824,NaN
1,Barbara,Brown,barbara.brown1@nurse-sample.example.com,Certified Registered Nurse Anesthetist,HCA Healthcare,"Birmingham, AL",Birmingham,AL,USA,232-230-2535,https://www.linkedin.com/in/barbara-brown/,https://www.facebook.com/100000000012286,M


In [26]:
import re
from pydantic import BaseModel, field_validator

class NurseRefiner(BaseModel):
    first_name: str
    phone_number: str

    @field_validator('phone_number')
    @classmethod
    def standardize_phone(cls, v: str) -> str:
        # Refinement: Strip everything but numbers and add +1
        numbers_only = re.sub(r'\D', '', v)
        return f"+1{numbers_only}"

# Test your new 'accurate' function
messy_nurse = {"first_name": "Elizabeth", "phone_number": "450-428-3286"}
refined_nurse = NurseRefiner(**messy_nurse)
print(refined_nurse.phone_number) # Output: +14504283286

+14504283286


In [27]:
import re
# Inside your logic:
def clean_phone(phone_str):
    if phone_str:
        return re.sub(r'\D', '', phone_str)
    return phone_str

In [28]:
import pandas as pd
from pydantic import BaseModel
from typing import Optional

# This represents the 'Person' model from your models.py
class Person(BaseModel):
    first_name: str
    last_name: str
    email: Optional[str] = None
    job_title: Optional[str] = None

# THE BRAIN: This is the logic from your merge.py
def merge_logic(existing_person: Person, new_data: dict) -> Person:
    """
    Updates the person ONLY if the new data isn't empty
    and matches the identity.
    """
    # Check if it's actually the same person (Basic identity check)
    if existing_person.first_name.lower() != new_data.get('first_name', '').lower():
        print(f"🛑 REJECTED: {new_data.get('first_name')} is NOT {existing_person.first_name}")
        return existing_person

    # If identity matches, update empty fields
    updated_person = existing_person.copy()
    if not updated_person.email and new_data.get('email'):
        updated_person.email = new_data['email']
        print(f"✅ SUCCESS: Updated email for {existing_person.first_name}")

    return updated_person

# --- TEST CASE: The 'Brown' Family ---

# 1. Create Elizabeth from your CSV
elizabeth = Person(first_name="Elizabeth", last_name="Brown", job_title="ICU Nurse")

# 2. Try to 'Enrich' Elizabeth using Barbara's data by mistake
barbara_data = {
    "first_name": "Barbara",
    "last_name": "Brown",
    "email": "barbara.b@healthcrew.com"
}

print("Attempting to merge Barbara's data into Elizabeth's record...")
result = merge_logic(elizabeth, barbara_data)

print("\n--- Final Record for Elizabeth ---")
print(result.dict())

Attempting to merge Barbara's data into Elizabeth's record...
🛑 REJECTED: Barbara is NOT Elizabeth

--- Final Record for Elizabeth ---
{'first_name': 'Elizabeth', 'last_name': 'Brown', 'email': None, 'job_title': 'ICU Nurse'}


/tmp/ipykernel_699/3425333029.py:47: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(result.dict())


In [29]:
import pandas as pd
import numpy as np
import re
from pydantic import BaseModel, field_validator, ConfigDict
from typing import Optional

# 1. THE BRAIN (Standardizing the rules)
class HealthCrewPolisher(BaseModel):
    model_config = ConfigDict(extra='ignore')

    first_name: str
    last_name: str
    email: Optional[str] = None
    job_title: Optional[str] = None
    phone: Optional[str] = None

    @field_validator('first_name', 'last_name')
    @classmethod
    def format_names(cls, v):
        if pd.isna(v) or v is None: return "Unknown"
        return str(v).strip().title()

    @field_validator('email')
    @classmethod
    def format_email(cls, v):
        if pd.isna(v) or v is None: return None
        return str(v).strip().lower()

    @field_validator('phone')
    @classmethod
    def clean_phone(cls, v):
        if pd.isna(v) or v is None: return None
        # Removing dashes/dots to make it 'Polished'
        return re.sub(r'\D', '', str(v))

# 2. LOAD DATA
try:
    df_raw = pd.read_csv(nurses_cleaned_data_path)
    print("✅ Successfully loaded nurses_cleaned.csv")
except:
    print("❌ Error: Make sure nurses_cleaned.csv is uploaded to Colab!")

# 3. RUN THE REFINERY
polished_records = []

for _, row in df_raw.iterrows():
    try:
        # Create a polished version of each row
        nurse = HealthCrewPolisher(
            first_name=row['first_name'],
            last_name=row['last_name'],
            email=row['email_address'], # Your CSV uses this column name
            job_title=row['job_title'],
            phone=row['phone_number']
        )
        polished_records.append(nurse.model_dump())
    except:
        continue

# 4. CREATE THE POLISHED TABLE
df_polished = pd.DataFrame(polished_records)

if df_polished.empty:
    print("❌ The refinery failed. Check your column names!")
else:
    # Deduplicate to find the unique individuals
    df_final = df_polished.drop_duplicates(subset=['first_name', 'last_name', 'job_title']).reset_index(drop=True)

    print("\n--- CLI SIMULATION COMPLETE ---")
    print(f"Processed: {len(df_raw)} raw rows")
    print(f"Produced: {len(df_final)} polished unique records")

    # Show a preview of your work
    print("\n--- PREVIEW OF POLISHED DATA ---")
    print(df_final[['first_name', 'last_name', 'phone']].head())

    # Save to your new file
    df_final.to_csv('CLI_SIMULATION_RESULTS.csv', index=False)
    print("\n✅ File 'CLI_SIMULATION_RESULTS.csv' is ready!")

✅ Successfully loaded nurses_cleaned.csv
❌ The refinery failed. Check your column names!


In [30]:
import pandas as pd
import re

# 1. LOAD THE DATA
try:
    df_raw = pd.read_csv(nurses_cleaned_data_path)
    print("✅ File Loaded: nurses_cleaned.csv")
except Exception as e:
    print(f"❌ ERROR: Could not find file. Error: {e}")

# 2. THE REFINERY (Direct Cleaning Logic)
def polish_data(df):
    # Make a copy so we don't mess up the original
    df_clean = df.copy()

    # REFINEMENT: Fix Names (Title Case)
    df_clean['first_name'] = df_clean['first_name'].fillna('Unknown').str.strip().str.title()
    df_clean['last_name'] = df_clean['last_name'].fillna('Unknown').str.strip().str.title()

    # REFINEMENT: Fix Emails (Lowercase)
    if 'email_address' in df_clean.columns:
        df_clean['email'] = df_clean['email_address'].fillna('').str.strip().str.lower()

    # REFINEMENT: Clean Phone Numbers (Digits only)
    if 'phone_number' in df_clean.columns:
        df_clean['phone'] = df_clean['phone_number'].fillna('').apply(lambda x: re.sub(r'\D', '', str(x)))

    # REFINEMENT: Default Job Title
    df_clean['job_title'] = df_clean['job_title'].fillna('Registered Nurse').str.strip().str.title()

    return df_clean

# 3. RUN THE POLISHING
df_polished = polish_data(df_raw)

# 4. DEDUPLICATE (The "Golden Record" Logic)
# This removes the 1,100+ duplicates to find unique nurses
df_final = df_polished.drop_duplicates(subset=['first_name', 'last_name', 'job_title']).reset_index(drop=True)

# 5. ARCHITECT'S REPORT
print("\n--- 🚀 DATA REFINERY SUCCESSFUL ---")
print(f"📊 Raw Rows Processed: {len(df_raw)}")
print(f"✨ Unique Records Created: {len(df_final)}")
print(f"🧬 Duplicates Removed: {len(df_raw) - len(df_final)}")

# Show a preview to confirm the formatting is perfect
print("\n--- PREVIEW OF YOUR WORK ---")
print(df_final[['first_name', 'last_name', 'phone', 'job_title']].head())

# Save the final file
df_final.to_csv('HEALTHCREW_GOLDEN_RECORDS.csv', index=False)
print("\n💾 Your polished file 'HEALTHCREW_GOLDEN_RECORDS.csv' is ready!")

✅ File Loaded: nurses_cleaned.csv

--- 🚀 DATA REFINERY SUCCESSFUL ---
📊 Raw Rows Processed: 5000
✨ Unique Records Created: 3897
🧬 Duplicates Removed: 1103

--- PREVIEW OF YOUR WORK ---
  first_name last_name       phone                               job_title
0  Elizabeth     Brown  4504283286                               Icu Nurse
1    Barbara     Brown  2322302535  Certified Registered Nurse Anesthetist
2  Charlotte     Davis  9338659928                 Np - Nurse Practitioner
3    Michael     Davis  2069773615  Certified Registered Nurse Anesthetist
4      Linda    Wilson  4209816514                   Rn - Registered Nurse

💾 Your polished file 'HEALTHCREW_GOLDEN_RECORDS.csv' is ready!
